# LawBot retrieval-augmented Q&A demo

Phase 3 builds a small local FAISS index from official India Code PDFs, retrieves legal text with MMR and an optional cross-encoder, then answers from retrieved context with inline source references. It also measures retrieval against held-out question references and demonstrates abstention.

**Educational prototype only.** These are historical IPC/CrPC and Constitution sources. The Supreme Court page describes its case summaries as explanatory aids, not part of the Court’s reasons or for use in legal proceedings. Always consult current primary sources and a qualified lawyer.

Run in Colab/Kaggle with a GPU for answer generation. The PDF and embedding downloads require network access. The notebook uses the Phase 2 adapter when present; otherwise it clearly falls back to the base Qwen model. Run the cells in order.


In [ ]:
# Locate project and install Phase 3 dependencies only; Phase 2 is optional here.
import os, sys, subprocess
from pathlib import Path
candidate_roots = ([Path(os.environ["LAWBOT_PROJECT_DIR"]).expanduser()]
                   if os.environ.get("LAWBOT_PROJECT_DIR") else [])
candidate_roots += [Path.cwd(), Path.cwd().parent, Path("/content/lawbot"), Path("/kaggle/working/lawbot")]
PROJECT_DIR = next((p.resolve() for p in candidate_roots if (p / "src" / "rag.py").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Make the lawbot/ project folder available, or set LAWBOT_PROJECT_DIR.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-rag.txt")], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project:", PROJECT_DIR)


In [ ]:
# Download official statute PDFs, extract/chunk them, and inspect chunk sizes.
import json, math, re, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from src.rag import build_corpus_chunks, print_chunk_statistics
docs = build_corpus_chunks(PROJECT_DIR, download=True)
stats = print_chunk_statistics(docs)
sizes = [max(1, round(len(d.page_content) / 4)) for d in docs]
plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=30, color="#2878B5", edgecolor="white")
plt.xlabel("Estimated tokens per chunk (characters / 4)"); plt.ylabel("Chunk count")
plt.title("LawBot source chunk sizes"); plt.tight_layout()
plot_path = PROJECT_DIR / "reports" / "chunk_statistics.png"
plot_path.parent.mkdir(parents=True, exist_ok=True); plt.savefig(plot_path, dpi=160); plt.show()


In [ ]:
# Build a normalized MiniLM embedding index. Keep the index local to this project.
from src.rag import build_faiss_index, load_reranker
vectorstore, embeddings, index_dir = build_faiss_index(docs, PROJECT_DIR)
print("Saved FAISS index to", index_dir)


In [ ]:
# Load Qwen with the Phase 2 LoRA adapter if it exists.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = PROJECT_DIR / "adapters" / "lawbot-qwen1.5b"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto", device_map="auto")
if ADAPTER_DIR.exists() and any(ADAPTER_DIR.iterdir()):
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    MODEL_LABEL = "Phase 2 fine-tuned adapter"
else:
    MODEL_LABEL = "base Qwen2.5-1.5B-Instruct (adapter not found)"
    print("NOTICE:", MODEL_LABEL, "; run finetune_lawbot.ipynb and copy its adapter to", ADAPTER_DIR)
print("Answer model:", MODEL_LABEL)


In [ ]:
# Optional cross-encoder improves ordering at extra download and inference cost.
USE_RERANKER = False
reranker = load_reranker() if USE_RERANKER else None
from src.rag import retrieve_with_mmr
ABSTAIN = "I don't have enough evidence in my sources to answer this"

def generate_text(messages, max_new_tokens=192):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def rewrite_question(history, question):
    if not history:
        return question
    context = " ".join(f"{role}: {content}" for role, content in history[-4:])
    prompt = ("Rewrite the latest user question as a standalone search query using the conversation context. "
              "Return only the query. Do not answer it. Conversation: " + context +
              f" User: {question} Standalone query:")
    return generate_text([{"role": "user", "content": prompt}], max_new_tokens=64).strip().strip("\"'")

def answer_question(question, history=None, k=5, min_score=0.30):
    history = history or []
    standalone = rewrite_question(history, question)
    hits = retrieve_with_mmr(standalone, vectorstore, embeddings, k=k, fetch_k=24, reranker=reranker)
    if not hits or hits[0]["score"] < min_score:
        answer = ABSTAIN
    else:
        context_parts, citation_map = [], {}
        for i, hit in enumerate(hits, 1):
            meta, content = hit["document"].metadata, hit["document"].page_content
            if meta.get("content_type") == "official_summary_paraphrase":
                cite = f"[{meta.get('case_citation', 'Supreme Court summary')}]"
            else:
                section = meta.get("section_number")
                if meta.get("act") == "Constitution":
                    cite = f"[Constitution Art. {section}]" if section else f"[Constitution pp. {meta.get('page')}]"
                else:
                    cite = f"[{meta.get('act')} section {section}]" if section else f"[{meta.get('act')} pp. {meta.get('page')}]"
            citation_map[cite] = meta.get("source_url")
            context_parts.append(f"SOURCE {i} {cite} (retrieval score {hit['score']:.3f}):\n{content}")
        system = ("Answer only from the supplied source excerpts. Do not add outside legal knowledge. "
                  f"If the excerpts do not directly support the requested answer, reply exactly: {ABSTAIN}. "
                  "Keep the answer concise. Cite every factual legal claim using one of the source labels shown. "
                  "For Supreme Court summary excerpts, say the source is an official summary, not the judgment.")
        user = f"Question: {standalone}\n\nSources:\n" + "\n\n---\n\n".join(context_parts)
        answer = generate_text([{"role": "system", "content": system}, {"role": "user", "content": user}])
        if ABSTAIN.lower() in answer.lower():
            answer = ABSTAIN
    return {"question": question, "standalone_query": standalone, "answer": answer, "hits": hits}


## Retrieval, citations, and abstention

The example starts with one question, then asks a follow-up. The follow-up is rewritten into a standalone retrieval query. The default score floor of 0.30 is only a demonstration setting; calibrate it on held-out queries below. Similarity scores are not probabilities and must not be treated as a legal-confidence guarantee.


In [ ]:
history = []
demo = answer_question("What does IPC section 302 provide?", history)
print("Query:", demo["standalone_query"])
print("Answer:\n", demo["answer"])
print("Retrieved evidence:")
for rank, item in enumerate(demo["hits"], 1):
    meta = item["document"].metadata
    print(f"{rank}. score={item['score']:.3f} {meta.get('act')} {meta.get('section_number') or meta.get('page')} | {meta.get('source_url')}")
history = [("user", "What does IPC section 302 provide?"), ("assistant", demo["answer"])]
followup = answer_question("And what about attempt?", history)
print("\nFollow-up rewritten as:", followup["standalone_query"])
print(followup["answer"])


In [ ]:
# Calibrate an illustrative score threshold from reference-bearing validation questions.
# This reports threshold performance on the same sampled items used to choose it; use a separate
# calibration split for an unbiased estimate. Unsupported prompts are synthetic stress cases.
import json, random
val_rows = [json.loads(line) for line in (PROJECT_DIR / "data/processed/val.jsonl").read_text(encoding="utf-8").splitlines()]
referenced = [r for r in val_rows if r.get("references")][:40]
random.Random(42).shuffle(referenced); referenced = referenced[:30]
positive_scores = []
for row in referenced:
    hits = retrieve_with_mmr(row["instruction"], vectorstore, embeddings, k=5, fetch_k=24, reranker=reranker)
    positive_scores.append(max((h["score"] for h in hits), default=0.0))
negative_questions = ["What is the current tax rate for a new cryptocurrency product?",
                      "Which law applies to a contract signed on Mars?",
                      "What is the current weather in Mumbai?",
                      "Give me the latest market price of gold today."]
negative_scores = [max((h["score"] for h in retrieve_with_mmr(q, vectorstore, embeddings, k=5, reranker=reranker)), default=0.0)
                   for q in negative_questions]
candidate_thresholds = sorted(set([0.0, 1.0] + positive_scores + negative_scores))
def balanced_accuracy(threshold):
    tpr = sum(s >= threshold for s in positive_scores) / max(1, len(positive_scores))
    tnr = sum(s < threshold for s in negative_scores) / max(1, len(negative_scores))
    return (tpr + tnr) / 2
CALIBRATED_THRESHOLD = max(candidate_thresholds, key=balanced_accuracy)
print({"positive_queries":len(positive_scores),"synthetic_negative_queries":len(negative_scores),
       "chosen_threshold":round(CALIBRATED_THRESHOLD,4),"sample_balanced_accuracy":round(balanced_accuracy(CALIBRATED_THRESHOLD),3),
       "caution":"Threshold selection scores are in-sample and synthetic negatives do not validate legal correctness."})


In [ ]:
# Retrieval evaluation: Hit@5 and reciprocal rank when an extracted section/article reference appears in metadata.
def reference_digits(reference):
    match = re.search(r"\b(\d{1,3}[A-Za-z]?)\b", reference)
    return match.group(1).upper() if match else None
eval_rows = [r for r in val_rows if r.get("references")][:100]
random.Random(17).shuffle(eval_rows); eval_rows = eval_rows[:30]
hits_at_5, reciprocal_ranks = [], []
for row in eval_rows:
    gold = {reference_digits(ref) for ref in row["references"]}
    gold.discard(None)
    retrieved = retrieve_with_mmr(row["instruction"], vectorstore, embeddings, k=5, fetch_k=24)
    ranks = [i for i, h in enumerate(retrieved, 1)
             if gold.intersection(set(h["document"].metadata.get("section_numbers", [])) |
                                  ({h["document"].metadata.get("section_number")} if h["document"].metadata.get("section_number") else set()))]
    hits_at_5.append(bool(ranks)); reciprocal_ranks.append(1 / min(ranks) if ranks else 0.0)
evaluation = {"n":len(eval_rows),"hit_at_5":round(sum(hits_at_5)/max(1,len(hits_at_5)),4),
              "mrr":round(sum(reciprocal_ranks)/max(1,len(reciprocal_ranks)),4),
              "metric_note":"Reference matching is numeric and noisy; it does not assess legal answer correctness."}
report = PROJECT_DIR / "reports" / "rag_eval_results.md"
report.write_text("# Phase 3 retrieval evaluation\n\n" + json.dumps(evaluation, indent=2) +
                  "\n\nScore threshold selected in the same calibration sample: " + str(CALIBRATED_THRESHOLD) +
                  ". Synthetic negatives and in-sample threshold selection make this an illustration, not a generalization estimate.\n", encoding="utf-8")
print(evaluation); print("Wrote", report)
